<a href="https://colab.research.google.com/github/Niviriti/dataviz-exercises-Nivritivytla/blob/main/lecture07_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lecture 7 — Class Exercise
## Heatmap & Waterfall: Netflix Catalogue

> **Push to:** `week07/lecture07_exercise.ipynb`

**Rules:**
1. Heatmap: colour scale must match the data type (sequential for counts, diverging for above/below)
2. Waterfall: use green for additions, red for subtractions, blue for totals
3. Insight title tells the setup-conflict-resolution story (or at minimum states the finding)
4. Annotate at least one cell or bar directly

---


In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv('/content/netflix_catalogue.csv')
print(f"Loaded: {len(df)} titles")
print(df['type'].value_counts())
print(df.head())


Loaded: 3000 titles
type
Movie      1974
TV Show    1026
Name: count, dtype: int64
      type  release_year  added_year             genre        country rating  \
0    Movie          2014        2016  Sci-Fi & Fantasy         France  PG-13   
1    Movie          2010        2014     Documentaries  United States  TV-MA   
2  TV Show          2011        2012     Kids & Family  United States  TV-14   
3    Movie          2016        2018             Anime          India     PG   
4    Movie          2014        2016     Kids & Family         Canada  TV-MA   

   duration  
0       157  
1       127  
2         6  
3       134  
4        77  


In [2]:
print("Genres:", df['genre'].value_counts().head(8))
print("\nCountries:", df['country'].value_counts().head(8))
print("\nRatings:", df['rating'].value_counts())


Genres: genre
Sports                244
Sci-Fi & Fantasy      213
Kids & Family         209
Crime                 206
Drama                 204
Horror                199
Action & Adventure    198
Thrillers             195
Name: count, dtype: int64

Countries: country
United States     932
India             337
United Kingdom    261
Japan             187
France            176
Canada            164
South Korea       151
Mexico            138
Name: count, dtype: int64

Ratings: rating
TV-MA    840
TV-14    733
PG-13    589
R        312
PG       196
TV-PG    128
G         92
TV-Y7     57
TV-G      53
Name: count, dtype: int64


## Task 1 — Heatmap: content by rating and release decade

**What to build:** A heatmap showing the number of titles by **content rating** (y-axis) and **decade** (x-axis).

**Requirements:**
- Create a 'decade' column: `df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'`
- Filter to TV-14, TV-MA, PG-13, R, PG only (most common ratings)
- Sequential colour scale (Blues)
- Values shown in cells (`text_auto=True`)
- Insight title about which rating dominates which decade


In [6]:
## Task 1 — Heatmap: Content by Rating and Release Decade

ratings_filter = ['TV-14', 'TV-MA', 'PG-13', 'R', 'PG']

filtered_df = df[df['rating'].isin(ratings_filter)]

heatmap_data = (
    filtered_df
    .groupby(['rating', 'decade'])
    .size()
    .reset_index(name='count')
)

pivot_table = heatmap_data.pivot(
    index='rating',
    columns='decade',
    values='count'
).fillna(0)

fig = px.imshow(
    pivot_table,
    text_auto=True,
    color_continuous_scale='Blues',
    aspect='auto',
    title='TV-MA titles dominate Netflix releases from the 2010s onward'
)

max_position = pivot_table.stack().idxmax()

fig.add_annotation(
    x=max_position[1],
    y=max_position[0],
    text='Highest concentration',
    showarrow=True,
    arrowhead=2
)

fig.update_layout(
    xaxis_title='Release Decade',
    yaxis_title='Content Rating'
)

fig.show()








Insights : TV-MA content is the most common on Netflix.Most content was released during the 2010s.Family-friendly ratings like PG are less frequent.Older decade releases are limited.

## Task 2 — Waterfall: Movie vs TV Show additions by year

**What to build:** A waterfall chart showing how Netflix's **Movie library** grew year by year (2015-2022).

**Requirements:**
- Filter to Movies only
- Group by `added_year`, count titles per year
- Final bar should be the cumulative total
- Green bars (additions), blue total
- Annotation on the year with the largest single addition
- Insight title naming the growth story


In [7]:
# Task 2

movies_df = df[df['type'] == 'Movie']

movies_yearly = (
    movies_df[
        (movies_df['added_year'] >= 2015) &
        (movies_df['added_year'] <= 2022)
    ]
    .groupby('added_year')
    .size()
    .reset_index(name='count')
)

max_row = movies_yearly.loc[movies_yearly['count'].idxmax()]

fig = go.Figure(
    go.Waterfall(
        measure=['relative'] * len(movies_yearly) + ['total'],
        x=[str(year) for year in movies_yearly['added_year']] + ['Total'],
        y=movies_yearly['count'].tolist() + [movies_yearly['count'].sum()],
        text=movies_yearly['count'].tolist() + [movies_yearly['count'].sum()],
        textposition='outside'
    )
)

fig.add_annotation(
    x=str(int(max_row['added_year'])),
    y=max_row['count'],
    text='Largest yearly addition',
    showarrow=True,
    arrowhead=2
)

fig.update_layout(
    title='Netflix movie catalogue surged during the late 2010s expansion phase',
    xaxis_title='Year Added',
    yaxis_title='Movies Added'
)

fig.show()

Insights : Netflix movie additions increased rapidly after 2015.One year shows the highest movie addition spike. Overall movie catalogue growth remained positive. The total bar highlights major platform expansion.